# Permutation Importance

## Teoría

Revisar página de Kaggle [Permutation Importance Kaggle](https://www.kaggle.com/code/dansbecker/permutation-importance)

Each row shows how much model performance decreased with a random shuffling 

**Note:** You'll occasionally see negative values for permutation importances. In those cases, the predictions on the shuffled (or noisy) data happened to be more accurate than the real data. This happens when the feature didn't matter (should have had an importance close to 0), but random chance caused the predictions on shuffled data to be more accurate. This is more common with small datasets

In [78]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import h5py
import json
import pickle
import os
import shap
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate

import eli5
from eli5.sklearn import PermutationImportance
from sklearn.inspection import permutation_importance


In [100]:
class CFG:
    colab = False  # Cambiar a True si se usa Colab
    Root = '..' if not colab else '/content/drive/MyDrive/PAI'
    path_df_imputed = f'{Root}/BaseDatos/df_imputed_with_original.csv'
    path_df_imputed_corrected = f'{Root}/BaseDatos/df_imputed_corrected.csv'
    elements_list = ["Nitrogen", "Phosphorus", "Potassium"]
    productivity_vars = ["Plant_Height (cm)", "Number of Flowers", 'Number of Harvested Fruits', 
                         'Weight of Harvested Fruits (Kg)','Fruit Height (mm)', 'Fruit Diameter (mm)']
    model_list = ['RF', 'SVM', 'MLP', 'KNN']
    include_prod = False  # Para incluir variables de productividad


    # NOTE: Cambiar el siguiente flag segun el tipo de entrenamiento
    individual_train = True # Para entrenar con los elementos por separado
    cuartiles_train = False

    # NOTE: Cambiar el siguiente path segun include prod
    if include_prod:
        if cuartiles_train:
            class_path = f'{Root}/Resultados/classification_cuartiles_include_prod/'
        else:
            class_path = f'{Root}/Resultados/classification_include_prod/'
    else:
        if cuartiles_train:
            class_path = f'{Root}/Resultados/classification_cuartiles_exclude_prod/'
        else:
            class_path = f'{Root}/Resultados/classification_exclude_prod/'


    if individual_train:
        path_pkl_results_classification = f"{class_path}class_results_individual_elements.pkl"
    elif cuartiles_train:
        path_pkl_results_classification = f"{class_path}class_models_cuartiles.pkl"
    else:
        path_pkl_results_classification = f"{class_path}all_classification_models.pkl"

    treat_quantiles_path = f'{Root}/Resultados/treatments_quantile_unified.json'

## Funciones

In [101]:
def codificar_clase(n, p, k):
    """Codificación base de 8 clases basada en combinaciones de nutrientes."""
    if n == 1 and p == 1 and k == 1:
        return 7
    elif n == 1 and p == 1:
        return 4
    elif n == 1 and k == 1:
        return 5
    elif p == 1 and k == 1:
        return 6
    elif n == 1:
        return 1
    elif p == 1:
        return 2
    elif k == 1:
        return 3
    else:
        return 0

def codificar_clase_personalizada(n, p, k, n_clases):
    """Codificación personalizada para 2-5 clases."""
    clase_base = codificar_clase(n, p, k)
    if n_clases == 2:
        return 1 if clase_base in [4, 5, 6, 7] else 0
    elif n_clases == 3:
        if clase_base in [1, 2, 3]:
            return 1
        elif clase_base in [4, 5, 6, 7]:
            return 2
        else:
            return 0
    elif n_clases == 4:
        return sum([n == 1, p == 1, k == 1])
    elif n_clases == 5:
        if n == 1 and p == 1 and k == 1:
            return 3
        elif (n == 1 and p == 1) or (n == 1 and k == 1) or (p == 1 and k == 1):
            return 2
        elif n == 1 or p == 1 or k == 1:
            return 1
        elif n == 0 and p == 0 and k == 0:
            return 0
        else:
            return 4
    else:
        return clase_base

def codificar_clase_6(c):
    """Agrupación para 6 clases."""
    if c == 0:
        return 0
    elif c == 1:
        return 1
    elif c in [2, 3]:
        return 2
    elif c in [4, 5]:
        return 3
    elif c == 6:
        return 4
    elif c == 7:
        return 5

def codificar_clase_7(c):
    """Agrupación para 7 clases."""
    if c in [5, 6]:
        return 5
    elif c == 7:
        return 6
    else:
        return c

def codificar_clase_nk_9(n, k):
    """Codificación de 9 clases basada en N y K (ignora P)."""
    if n not in [0, 1, 2] or k not in [0, 1, 2]:
        return None
    return 3 * n + k

def codificar_clase_agrupada(n, p, k, n_clases):
    """Función principal de codificación según número de clases."""
    base = codificar_clase(n, p, k)
    if n_clases == 5:
        return codificar_clase_personalizada(n, p, k, 5)
    elif n_clases == 6:
        return codificar_clase_6(base)
    elif n_clases == 7:
        return codificar_clase_7(base)
    elif n_clases == 8:
        return base
    elif n_clases == 9:
        return codificar_clase_nk_9(n, k)
    else:
        return codificar_clase_personalizada(n, p, k, n_clases)

def codificar_clase_individual(df, element):
    if element not in CFG.elements_list:
        raise ValueError(f"Elemento '{element}' no válido. Debe ser uno de {CFG.elements_list}.")
    return df[element].copy()

def codificar_clase_cuartiles(df, filter_data=True):
    """
    Codifica clases basado en cuartiles de productividad.
    
    Args:
        df: DataFrame con los datos
        filter_data: Si True, elimina tratamientos que no están en Q1 o Q4
    
    Returns:
        Series con las clases codificadas (y DataFrame filtrado si filter_data=True)
    """
    # Leer json con tratamientos asociados a cuartiles
    with open(CFG.treat_quantiles_path, 'r') as f:
        treatments_quantile_unified = json.load(f)
    
    # Crear diccionario inverso: {treatment_num: clase}
    treatment_to_class = {}
    for clase, treatments in treatments_quantile_unified.items():
        for treatment in treatments:
            treatment_to_class[treatment] = int(clase)
    
    if filter_data:
        # Filtrar solo tratamientos Q1 y Q4
        valid_treatments = list(treatment_to_class.keys())
        df_filtered = df[df['Treatment_Num'].isin(valid_treatments)].copy()
        
        # Mapear a clases y remapear a 0 y 1
        y = df_filtered['Treatment_Num'].map(treatment_to_class)
        
        return df_filtered, y
    else:
        # Solo mapear sin filtrar (generará NaN)
        y = df['Treatment_Num'].map(treatment_to_class)
        return y

In [102]:
def preparar_datos(df_imputed, n_clases=3, element="Nitrogen", test_size=0.3, random_state=42):
    """Prepara los datos para entrenamiento.

    Args:
        df_imputed (DataFrame): DataFrame con datos imputados.
        n_clases (int): Número de clases para la codificación.
        element (str): Elemento a utilizar para la codificación individual.
        test_size (float): Proporción del conjunto de prueba.
        random_state (int): Semilla para reproducibilidad.
    Returns:
        tuple: (X_train, X_test, y_train, y_test, feature_names, class_distribution)
    """
    df = df_imputed.copy()
    
    # NOTE: Ahroa se trabajan con 3 clases
    if CFG.individual_train:
        df['target'] = codificar_clase_individual(df, element=element)
    elif CFG.cuartiles_train:
        df, df['target'] = codificar_clase_cuartiles(df, filter_data=True)
    else:
    # NOTE: La siguiente codificación se hacía cuando se trataban 2 a 9 clases
        df['target'] = df.apply(
            lambda row: codificar_clase_agrupada(
                row['Nitrogen'], row['Phosphorus'], row['Potassium'], n_clases
            ), axis=1
        )
    

    # Distribución de clases
    class_distribution = df['target'].value_counts().sort_index()

    # Eliminar columnas no necesarias
    columns_to_drop = ['Nitrogen', 'Phosphorus', 'Potassium', 'target',
                       'Clase_custom', 'Treatment_Num', 'Year', 'Month', 'Day']
    # Eliminar variables de productividad si no se desean incluir
    if not CFG.include_prod:
        columns_to_drop += CFG.productivity_vars
    X = df.drop(columns=columns_to_drop, errors='ignore')
    y = df['target']

    # División de datos
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, stratify=y, test_size=test_size, random_state=random_state
    )

    return X_train, X_test, y_train, y_test, X.columns, class_distribution

## Cargar Modelos

In [103]:
# load from pickle
with open(CFG.path_pkl_results_classification, 'rb') as f:
    class_results = pickle.load(f)

In [104]:
class_results['RF'][0]['grid_search'].best_estimator_

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 RandomForestClassifier(max_depth=20, n_estimators=300,
                                        random_state=42))])

In [105]:
def clean_feature_names(feature_names):
    """Remove characters that XGBoost doesn't allow in feature names."""
    cleaned = []
    for name in feature_names:
        # Replace brackets with parentheses or underscores
        cleaned_name = name.replace('[', '(').replace(']', ')').replace('<', '_lt_')
        cleaned.append(cleaned_name)
    return cleaned

In [106]:
df_imputed = pd.read_csv(CFG.path_df_imputed_corrected)
df_imputed.columns = clean_feature_names(df_imputed.columns)


In [127]:
model_mlp = class_results['MLP'][2]['grid_search'].best_estimator_
print(model_mlp.steps)


[('scaler', StandardScaler()), ('clf', MLPClassifier(alpha=1e-05, hidden_layer_sizes=(100, 50), max_iter=500,
              random_state=42))]


In [130]:
model_ejemplo = class_results['XGB'][0]['grid_search'].best_estimator_
X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
                df_imputed, n_clases=3, element="Nitrogen"
            )
explainer = shap.TreeExplainer(model_ejemplo)

# calculate shap values. This is what we will plot.
shap_values = explainer.shap_values(X_train)

shap.dependence_plot('Soil NO3 Horiba (ppm)', shap_values[1], X_train, interaction_index="7in1_Conductivity(mS/cm)")



InvalidModelError: Model type not yet supported by TreeExplainer: <class 'sklearn.pipeline.Pipeline'>

### Ejemplo con modelo

In [107]:
#ejemplo para un elemento y algoritmo
element = "Nitrogen"
X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
    df_imputed, n_clases=3, element=element
)
model = class_results['RF'][0]['grid_search'].best_estimator_
perm = PermutationImportance(model, scoring="accuracy",random_state=0).fit(X_test, y_test)

In [108]:
# Lista de cambio del resultado de métrica
print(perm.feature_importances_)
print((perm.feature_importances_).shape)

[0.04069767 0.01017442 0.03430233 0.02936047 0.0377907  0.03517442
 0.03546512 0.04215116 0.01104651 0.01627907 0.01947674 0.05290698
 0.00813953 0.01075581 0.00581395 0.00988372 0.00465116 0.00959302
 0.00784884 0.01017442]
(20,)


In [109]:
# si es negativo, volver a 0, convertir a porcentaje
perm_importances = perm.feature_importances_
perm_importances[perm_importances < 0] = 0
perm_importances = perm_importances / np.sum(perm_importances)
print(perm_importances)

[0.09427609 0.02356902 0.07946128 0.06801347 0.08754209 0.08148148
 0.08215488 0.0976431  0.02558923 0.03771044 0.04511785 0.12255892
 0.01885522 0.02491582 0.01346801 0.02289562 0.01077441 0.02222222
 0.01818182 0.02356902]


## Permutation Importance - 3 Classes (codificado por elemento)

In [110]:
if CFG.individual_train:
    algorithms = class_results.keys()
    list_elements = ['Nitrogen', 'Phosphorus', 'Potassium']
    for alg in algorithms:
        for i in range(len(list_elements)):
            X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
                df_imputed, n_clases=3, element=list_elements[i]
            )
            model = class_results[alg][i]['grid_search'].best_estimator_
            perm = PermutationImportance(model, random_state=0, n_iter = 15).fit(X_test, y_test)
            
            print(f'Permutation Importance for {alg} - {list_elements[i]}:')
            #print(eli5.format_as_text(eli5.explain_weights(perm, feature_names = X_test.columns.tolist())))
            display(eli5.show_weights(perm, feature_names = X_test.columns.tolist()))

Permutation Importance for RF - Nitrogen:


Weight,Feature
0.0504 ± 0.0167,Soil NO3 Horiba (ppm)
0.0405 ± 0.0153,Soil pH Horiba
0.0397 ± 0.0115,Chlorophyll (SPAD)
0.0352 ± 0.0160,Sap Na (ppm)
0.0317 ± 0.0147,Sap Ca (ppm)
0.0314 ± 0.0138,Sap NO3 (ppm)
0.0294 ± 0.0185,Sap Conductivity (mS/cm)
0.0293 ± 0.0152,Sap K (ppm)
0.0192 ± 0.0137,Soil Na Horiba (ppm)
0.0177 ± 0.0078,Soil Ca Horiba (ppm)


Permutation Importance for RF - Phosphorus:


Weight,Feature
0.0407 ± 0.0128,Sap NO3 (ppm)
0.0297 ± 0.0166,Soil K Horiba (ppm)
0.0229 ± 0.0133,Sap Na (ppm)
0.0164 ± 0.0112,Soil NO3 Horiba (ppm)
0.0097 ± 0.0173,Soil pH Horiba
0.0083 ± 0.0112,Chlorophyll (SPAD)
0.0083 ± 0.0145,Sap Conductivity (mS/cm)
0.0060 ± 0.0117,Soil Na Horiba (ppm)
0.0060 ± 0.0086,Sap Ca (ppm)
0.0044 ± 0.0074,Sap K (ppm)


Permutation Importance for RF - Potassium:


...

Weight,Feature
0.0564 ± 0.0155,Soil NO3 Horiba (ppm)
0.0489 ± 0.0145,Sap Na (ppm)
0.0453 ± 0.0124,Sap Ca (ppm)
0.0352 ± 0.0188,Sap Conductivity (mS/cm)
0.0351 ± 0.0149,Sap NO3 (ppm)
0.0317 ± 0.0182,Sap K (ppm)
0.0250 ± 0.0100,Soil pH Horiba
0.0208 ± 0.0152,Soil K Horiba (ppm)
0.0197 ± 0.0148,Chlorophyll (SPAD)
0.0176 ± 0.0117,Soil Na Horiba (ppm)


Permutation Importance for XGB - Phosphorus:


Weight,Feature
0.0720 ± 0.0174,Sap NO3 (ppm)
0.0565 ± 0.0154,Soil K Horiba (ppm)
0.0425 ± 0.0116,Soil Ca Horiba (ppm)
0.0320 ± 0.0144,Soil pH Horiba
0.0319 ± 0.0122,Soil Na Horiba (ppm)
0.0310 ± 0.0111,Sap Conductivity (mS/cm)
0.0303 ± 0.0147,Sap Na (ppm)
0.0207 ± 0.0151,Sap K (ppm)
0.0191 ± 0.0126,Soil NO3 Horiba (ppm)
0.0181 ± 0.0103,Sap pH


Permutation Importance for XGB - Potassium:


Weight,Feature
0.0998 ± 0.0243,Sap Ca (ppm)
0.0668 ± 0.0180,Sap Conductivity (mS/cm)
0.0482 ± 0.0179,Sap K (ppm)
0.0476 ± 0.0148,Sap Na (ppm)
0.0403 ± 0.0118,Soil K Horiba (ppm)
0.0396 ± 0.0208,Soil pH Horiba
0.0393 ± 0.0211,Sap NO3 (ppm)
0.0314 ± 0.0210,Chlorophyll (SPAD)
0.0305 ± 0.0128,Soil Conductivity Horiba (mS/cm)
0.0275 ± 0.0131,Sap pH


In [ ]:
def permutation_importance_all_elements(class_results, df_imputed, dir_path_permutation, list_elements):
    
    algorithms = class_results.keys()
    dict_perm_impt_vals = {}
    for i in range(len(list_elements)):
        df = pd.DataFrame()
        dict_perm_impt_vals[list_elements[i]] = {}
        for alg in algorithms:
            X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
                df_imputed, n_clases=3, element=list_elements[i]
            )
            model = class_results[alg][i]['grid_search'].best_estimator_
            perm = PermutationImportance(model, random_state=0, n_iter=15, cv="prefit").fit(X_test, y_test)
            
            print(f'Permutation Importance for {alg} - {list_elements[i]}:')
            
            perm_importances = perm.feature_importances_
            dict_perm_impt_vals[list_elements[i]][alg] = perm_importances
            #agregar a df
            df[alg] = perm_importances
        #agregar nombres de features en primera columna
        df.insert(0, 'Feature', X_test.columns.tolist())
        #display(df)
        #guardar
        df.to_csv(f'{dir_path_permutation}permutation_importance_{list_elements[i]}.csv', index=False)
    return dict_perm_impt_vals


def most_frequent_variables_analysis(df_imputed, dict_perm_impt_vals, dir_path_permutation):
    percentages = [0.7, 0.8]
    for percentage in percentages:
        for element in dict_perm_impt_vals: #recorre cada modelo (elementos o cuartil)
            best_feats_element = {}
            for alg in dict_perm_impt_vals[element]:
                vals = dict_perm_impt_vals[element][alg]
                vals[vals < 0] = 0
                vals = vals / np.sum(vals)
                X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
                df_imputed, n_clases=3, element=element
                )
                
                # Revisar porcentaje acumulado
                df = pd.DataFrame({'Variable': X_test.columns.tolist(), 'Importance': vals})
                df = df.sort_values(by='Importance', ascending=False)
                df['Cumulative_Importance'] = df['Importance'].cumsum()

                top_feats = df[df['Cumulative_Importance'] <= percentage]
                #guardar el nombre de las variables en diccionario
                best_feats_element[alg] = top_feats['Variable'].tolist()
            
            # guardar csv
            df_best_feats = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in best_feats_element.items()]))
            print(f"{element}")
            display(df_best_feats)
            df_best_feats.to_csv(f'{dir_path_permutation}best_{int(percentage*100)}_percent_features_{element}.csv', index=False)


In [112]:
if CFG.individual_train:
    dir_path_permutation = f'{CFG.class_path}permutation_importance/'
    os.makedirs(dir_path_permutation, exist_ok=True)
    dict_perm_impt_vals = permutation_importance_all_elements(class_results, df_imputed, dir_path_permutation,
                                                              list_elements = ['Nitrogen', 'Phosphorus', 'Potassium'])


Permutation Importance for RF - Nitrogen:
Permutation Importance for SVM - Nitrogen:
Permutation Importance for KNN - Nitrogen:
Permutation Importance for MLP - Nitrogen:
Permutation Importance for XGB - Nitrogen:
Permutation Importance for RF - Phosphorus:
Permutation Importance for SVM - Phosphorus:
Permutation Importance for KNN - Phosphorus:
Permutation Importance for MLP - Phosphorus:
Permutation Importance for XGB - Phosphorus:
Permutation Importance for RF - Potassium:
Permutation Importance for SVM - Potassium:
Permutation Importance for KNN - Potassium:
Permutation Importance for MLP - Potassium:
Permutation Importance for XGB - Potassium:


In [ ]:
if CFG.individual_train:

    most_frequent_variables_analysis(df_imputed, dict_perm_impt_vals, dir_path_permutation)

Nitrogen


,RF,SVM,KNN,MLP,XGB
0,Soil NO3 Horiba (ppm),Sap Na (ppm),Sap Ca (ppm),Sap NO3 (ppm),Soil NO3 Horiba (ppm)
1,Soil pH Horiba,Sap Ca (ppm),Sap Na (ppm),Soil NO3 Horiba (ppm),Sap Na (ppm)
2,Chlorophyll (SPAD),Sap NO3 (ppm),Sap NO3 (ppm),Sap Na (ppm),Sap Ca (ppm)
3,Sap Na (ppm),Soil NO3 Horiba (ppm),Soil pH Horiba,Sap Ca (ppm),Sap Conductivity (mS/cm)
4,Sap Ca (ppm),7in1_S_Temperature(C),Sap Conductivity (mS/cm),Soil pH Horiba,Sap NO3 (ppm)
5,Sap NO3 (ppm),Air_sensor_Temperature(C),Pynamometer_Radiation(W/m2),Soil Na Horiba (ppm),Sap K (ppm)
6,Sap Conductivity (mS/cm),Soil pH Horiba,Soil Ca Horiba (ppm),Soil Ca Horiba (ppm),Soil pH Horiba
7,Sap K (ppm),Chlorophyll (SPAD),Chlorophyll (SPAD),7in1_S_Temperature(C),Soil K Horiba (ppm)
8,NaN,Sap Conductivity (mS/cm),7in1_Moisture(%RH),Chlorophyll (SPAD),NaN
9,NaN,Air_sensor_Humidity(%RH),NaN,Sap Conductivity (mS/cm),NaN


Phosphorus


,RF,SVM,KNN,MLP,XGB
0,Sap NO3 (ppm),Sap NO3 (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap NO3 (ppm)
1,Soil K Horiba (ppm),Soil Ca Horiba (ppm),Sap Na (ppm),Soil Ca Horiba (ppm),Soil K Horiba (ppm)
2,Sap Na (ppm),Sap Na (ppm),Sap NO3 (ppm),7in1_S_Temperature(C),Soil Ca Horiba (ppm)
3,Soil NO3 Horiba (ppm),Soil Na Horiba (ppm),Sap Ca (ppm),Sap Na (ppm),Soil pH Horiba
4,NaN,Air_sensor_Temperature(C),Sap Conductivity (mS/cm),Soil Na Horiba (ppm),Soil Na Horiba (ppm)
5,NaN,Sap Conductivity (mS/cm),Pynamometer_Radiation(W/m2),Soil pH Horiba,Sap Conductivity (mS/cm)
6,NaN,Chlorophyll (SPAD),7in1_Moisture(%RH),Chlorophyll (SPAD),Sap Na (ppm)
7,NaN,Air_sensor_Humidity(%RH),Soil Na Horiba (ppm),Pynamometer_Radiation(W/m2),Sap K (ppm)
8,NaN,Soil NO3 Horiba (ppm),NaN,NaN,NaN


Potassium


...

,RF,SVM,KNN,MLP,XGB
0,Soil NO3 Horiba (ppm),Sap Na (ppm),Sap Ca (ppm),Sap NO3 (ppm),Soil NO3 Horiba (ppm)
1,Soil pH Horiba,Sap Ca (ppm),Sap Na (ppm),Soil NO3 Horiba (ppm),Sap Na (ppm)
2,Chlorophyll (SPAD),Sap NO3 (ppm),Sap NO3 (ppm),Sap Na (ppm),Sap Ca (ppm)
3,Sap Na (ppm),Soil NO3 Horiba (ppm),Soil pH Horiba,Sap Ca (ppm),Sap Conductivity (mS/cm)
4,Sap Ca (ppm),7in1_S_Temperature(C),Sap Conductivity (mS/cm),Soil pH Horiba,Sap NO3 (ppm)
5,Sap NO3 (ppm),Air_sensor_Temperature(C),Pynamometer_Radiation(W/m2),Soil Na Horiba (ppm),Sap K (ppm)
6,Sap Conductivity (mS/cm),Soil pH Horiba,Soil Ca Horiba (ppm),Soil Ca Horiba (ppm),Soil pH Horiba
7,Sap K (ppm),Chlorophyll (SPAD),Chlorophyll (SPAD),7in1_S_Temperature(C),Soil K Horiba (ppm)
8,Soil Na Horiba (ppm),Sap Conductivity (mS/cm),7in1_Moisture(%RH),Chlorophyll (SPAD),Chlorophyll (SPAD)
9,Soil Ca Horiba (ppm),Air_sensor_Humidity(%RH),7in1_Ph(pH),Sap Conductivity (mS/cm),Soil Na Horiba (ppm)


Phosphorus


,RF,SVM,KNN,MLP,XGB
0,Sap NO3 (ppm),Sap NO3 (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap NO3 (ppm)
1,Soil K Horiba (ppm),Soil Ca Horiba (ppm),Sap Na (ppm),Soil Ca Horiba (ppm),Soil K Horiba (ppm)
2,Sap Na (ppm),Sap Na (ppm),Sap NO3 (ppm),7in1_S_Temperature(C),Soil Ca Horiba (ppm)
3,Soil NO3 Horiba (ppm),Soil Na Horiba (ppm),Sap Ca (ppm),Sap Na (ppm),Soil pH Horiba
4,Soil pH Horiba,Air_sensor_Temperature(C),Sap Conductivity (mS/cm),Soil Na Horiba (ppm),Soil Na Horiba (ppm)
5,Chlorophyll (SPAD),Sap Conductivity (mS/cm),Pynamometer_Radiation(W/m2),Soil pH Horiba,Sap Conductivity (mS/cm)
6,NaN,Chlorophyll (SPAD),7in1_Moisture(%RH),Chlorophyll (SPAD),Sap Na (ppm)
7,NaN,Air_sensor_Humidity(%RH),Soil Na Horiba (ppm),Pynamometer_Radiation(W/m2),Sap K (ppm)
8,NaN,Soil NO3 Horiba (ppm),Chlorophyll (SPAD),Sap Conductivity (mS/cm),Soil NO3 Horiba (ppm)
9,NaN,Soil pH Horiba,Air_sensor_Humidity(%RH),Sap Ca (ppm),Sap pH


Potassium


,RF,SVM,KNN,MLP,XGB
0,Sap Ca (ppm),Sap Conductivity (mS/cm),Sap Ca (ppm),Sap Ca (ppm),Sap Ca (ppm)
1,Sap K (ppm),Sap NO3 (ppm),Sap Conductivity (mS/cm),Sap Conductivity (mS/cm),Sap Conductivity (mS/cm)
2,Sap Conductivity (mS/cm),Sap Ca (ppm),Sap NO3 (ppm),Sap Na (ppm),Sap K (ppm)
3,Sap NO3 (ppm),Sap Na (ppm),Sap Na (ppm),Sap NO3 (ppm),Sap Na (ppm)
4,Soil pH Horiba,Soil pH Horiba,Soil pH Horiba,Air_sensor_Humidity(%RH),Soil K Horiba (ppm)
5,Soil K Horiba (ppm),7in1_S_Temperature(C),Soil Na Horiba (ppm),Soil pH Horiba,Soil pH Horiba
6,Sap Na (ppm),Air_sensor_Humidity(%RH),7in1_S_Temperature(C),Soil Na Horiba (ppm),Sap NO3 (ppm)
7,7in1_Ph(pH),Sap K (ppm),Chlorophyll (SPAD),Pynamometer_Radiation(W/m2),Chlorophyll (SPAD)
8,NaN,Soil Na Horiba (ppm),Pynamometer_Radiation(W/m2),Soil NO3 Horiba (ppm),Soil Conductivity Horiba (mS/cm)
9,NaN,Soil Ca Horiba (ppm),7in1_Ph(pH),Soil K Horiba (ppm),Sap pH


## Permutation Importance - 2 Classes (codificado por Cuartiles)

In [ ]:
if CFG.cuartiles_train:
    dir_path_permutation = f'{CFG.class_path}permutation_importance/'
    os.makedirs(dir_path_permutation, exist_ok=True)
    dict_perm_impt_vals = permutation_importance_all_elements(class_results, df_imputed, dir_path_permutation,
                                                              list_elements = ['Quartiles'])
    most_frequent_variables_analysis(df_imputed, dict_perm_impt_vals, dir_path_permutation)

Permutation Importance for RF - Quartiles:
Permutation Importance for SVM - Quartiles:
Permutation Importance for KNN - Quartiles:
Permutation Importance for MLP - Quartiles:
Permutation Importance for XGB - Quartiles:
Quartiles


,RF,SVM,KNN,MLP,XGB
0,Sap NO3 (ppm),Sap NO3 (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap NO3 (ppm)
1,Soil K Horiba (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap Ca (ppm),Soil K Horiba (ppm)
2,Chlorophyll (SPAD),Sap Na (ppm),Sap K (ppm),Air_sensor_Temperature(C),Sap Na (ppm)
3,NaN,Sap Ca (ppm),Sap Na (ppm),Soil Ca Horiba (ppm),NaN
4,NaN,Sap K (ppm),Sap Ca (ppm),Sap Na (ppm),NaN
5,NaN,Pynamometer_Radiation(W/m2),7in1_Moisture(%RH),Chlorophyll (SPAD),NaN
6,NaN,Soil Ca Horiba (ppm),Air_sensor_Humidity(%RH),Air_sensor_Humidity(%RH),NaN
7,NaN,Air_sensor_Humidity(%RH),Pynamometer_Radiation(W/m2),Soil pH Horiba,NaN
8,NaN,7in1_Moisture(%RH),NaN,7in1_Moisture(%RH),NaN
9,NaN,NaN,NaN,Sap K (ppm),NaN


Quartiles


,RF,SVM,KNN,MLP,XGB
0,Sap NO3 (ppm),Sap NO3 (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap NO3 (ppm)
1,Soil K Horiba (ppm),Soil pH Horiba,Sap NO3 (ppm),Sap Ca (ppm),Soil K Horiba (ppm)
2,Chlorophyll (SPAD),Sap Na (ppm),Sap K (ppm),Air_sensor_Temperature(C),Sap Na (ppm)
3,Sap Na (ppm),Sap Ca (ppm),Sap Na (ppm),Soil Ca Horiba (ppm),Chlorophyll (SPAD)
4,NaN,Sap K (ppm),Sap Ca (ppm),Sap Na (ppm),NaN
5,NaN,Pynamometer_Radiation(W/m2),7in1_Moisture(%RH),Chlorophyll (SPAD),NaN
6,NaN,Soil Ca Horiba (ppm),Air_sensor_Humidity(%RH),Air_sensor_Humidity(%RH),NaN
7,NaN,Air_sensor_Humidity(%RH),Pynamometer_Radiation(W/m2),Soil pH Horiba,NaN
8,NaN,7in1_Moisture(%RH),Chlorophyll (SPAD),7in1_Moisture(%RH),NaN
9,NaN,Chlorophyll (SPAD),Sap Conductivity (mS/cm),Sap K (ppm),NaN


In [ ]:
if CFG.cuartiles_train:
    algorithms = class_results.keys()
    list_cuartiles = ["Q1", "Q4"]
    for alg in algorithms:
        for i in range(len(list_elements)):
            X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
                df_imputed, n_clases=4, element=list_elements[i]
            )
            model = class_results[alg][i]['grid_search'].best_estimator_
            perm = PermutationImportance(model, random_state=1).fit(X_test, y_test)
            print(f'Permutation Importance for {alg} - {list_elements[i]}:')
            display(eli5.show_weights(perm, feature_names = X_test.columns.tolist()))

### Permutation Importance Sklearn

In [27]:
from sklearn.inspection import permutation_importance
element = 'Nitrogen'  # Cambiar según el elemento a analizar
n_clases = 3  # Cambiar según el número de clases deseado
X_train, X_test, y_train, y_test, feature_names, class_dist = preparar_datos(
        df_imputed, n_clases, element=element
    )
model = class_results['RF'][0]['grid_search'].best_estimator_
result = permutation_importance(model, X_test, y_test, n_repeats=5,random_state=0)
perm_imp = result.importances_mean

perm_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': perm_imp
}).sort_values(by='Importance', ascending=False)
display(perm_imp_df)


,Feature,Importance
7,Soil pH Horiba,0.044477
11,Soil NO3 Horiba (ppm),0.044186
0,Chlorophyll (SPAD),0.039826
3,Sap Ca (ppm),0.034593
6,Sap Conductivity (mS/cm),0.030233
4,Sap Na (ppm),0.029651
5,Sap NO3 (ppm),0.026453
2,Sap K (ppm),0.025291
10,Soil Na Horiba (ppm),0.018895
13,7in1_Ph[pH],0.015988
